In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!pip install kaggle shap xgboost --quiet

In [ ]:
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store
!unzip -q ecommerce-behavior-data-from-multi-category-store.zip

## 2. Load & Optimise Memory

In [ ]:
import pandas as pd

df = pd.read_csv('2019-Oct.csv', nrows=500000)

# Downcast to reduce memory
df['event_type']    = df['event_type'].astype('category')
df['category_code'] = df['category_code'].astype('category')
df['brand']         = df['brand'].astype('category')
df['product_id']    = pd.to_numeric(df['product_id'],  downcast='integer')
df['category_id']   = pd.to_numeric(df['category_id'], downcast='integer')
df['user_id']       = pd.to_numeric(df['user_id'],     downcast='integer')
df['price']         = pd.to_numeric(df['price'],       downcast='float')
df['event_time']    = pd.to_datetime(df['event_time'])

print(df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Nulls:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nUnique users:', df['user_id'].nunique())
print('Unique sessions:', df['user_session'].nunique())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['event_type'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Event Type Distribution')
axes[0].set_xlabel('Event Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['hour'] = df['event_time'].dt.hour
df['hour'].value_counts().sort_index().plot(kind='line', marker='o', ax=axes[1])
axes[1].set_title('Shopping Activity by Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Number of Events')

plt.tight_layout()
plt.show()

In [ ]:
# Conversion funnel
event_counts = df['event_type'].value_counts()
views, carts, purchases = event_counts['view'], event_counts['cart'], event_counts['purchase']

print(f'View → Cart:     {carts / views * 100:.2f}%')
print(f'Cart → Purchase: {purchases / carts * 100:.2f}%')
print(f'View → Purchase: {purchases / views * 100:.2f}%')

In [ ]:
# Top categories by events vs purchases
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

df['category_code'].value_counts().head(10).plot(kind='bar', ax=axes[0])
axes[0].set_title('Top 10 Categories by Events')
axes[0].tick_params(axis='x', rotation=75)

df[df['event_type'] == 'purchase']['category_code'].value_counts().head(10).plot(kind='bar', ax=axes[1])
axes[1].set_title('Top 10 Categories by Purchases')
axes[1].tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
session_event_counts = df.groupby('user_session').size()
upper_limit = session_event_counts.quantile(0.999)
valid_sessions = session_event_counts[session_event_counts <= upper_limit].index
df = df[df['user_session'].isin(valid_sessions)]
print(f'Sessions after bot filter: {df["user_session"].nunique()} (removed {len(session_event_counts) - len(valid_sessions)} outlier sessions)')

In [ ]:
# ── LEAKAGE FIX ──────────────────────────────────────────────────────────
# Purchase events contain the answer (target). Using them in feature
# engineering tells the model which sessions purchased — data leakage.
# Rule: ALL features must be computed from view+cart events ONLY.
# df      = full data  (used only for target label creation)
# df_feat = view+cart only (used for every feature computation)
df_feat = df[df['event_type'].isin(['view', 'cart'])].copy()
print(f'df rows: {len(df):,}  →  df_feat rows (view+cart only): {len(df_feat):,}')
print(f'Purchase events excluded: {len(df) - len(df_feat):,}')

In [ ]:
df = df.sort_values(by=['user_session', 'event_time'])

In [ ]:
# --- Session time bounds ---
session_time = df.groupby('user_session')['event_time'].agg(['min', 'max'])
session_time['session_duration'] = (
    session_time['max'] - session_time['min']
).dt.total_seconds()

In [ ]:
# --- Target label ---
purchase_sessions = df[df['event_type'] == 'purchase']['user_session'].unique()
session_time['purchased'] = session_time.index.isin(purchase_sessions)
session_time['target']    = session_time['purchased'].astype(int)

print('Class distribution:')
print(session_time['target'].value_counts())
print(f'Purchase rate: {session_time["target"].mean() * 100:.2f}%')

In [ ]:
# --- Event-type counts per session ---
# Using df_feat (view+cart only) — purchase events excluded
for event in ['view', 'cart']:
    counts = df_feat[df_feat['event_type'] == event].groupby('user_session').size()
    session_time[f'{event}_count'] = counts
    session_time[f'{event}_count'] = session_time[f'{event}_count'].fillna(0)

# session_length = view+cart events only (no purchase events)
session_time['session_length']  = df_feat.groupby('user_session').size().reindex(session_time.index).fillna(0)
session_time['unique_products'] = df_feat.groupby('user_session')['product_id'].nunique().reindex(session_time.index).fillna(0)


In [ ]:
# --- Ratio features ---
session_time['cart_view_ratio'] = (
    session_time['cart_count'] / session_time['view_count']
).replace([float('inf')], 0).fillna(0)

In [ ]:
# --- Average time gap between events (view+cart only) ---
df_feat = df_feat.sort_values(by=['user_session', 'event_time'])
df_feat['time_diff'] = (
    df_feat.groupby('user_session')['event_time']
    .diff()
    .dt.total_seconds()
)
session_time['avg_time_gap'] = (
    df_feat.groupby('user_session')['time_diff'].mean()
).reindex(session_time.index).fillna(0)


In [ ]:
# --- Repeat product views (view events only) ---
total_views    = df_feat[df_feat['event_type'] == 'view'].groupby(['user_session', 'product_id']).size()
repeated_views = total_views[total_views > 1]
repeat_count   = repeated_views.groupby('user_session').size()
session_time['repeat_product_views'] = repeat_count.reindex(session_time.index).fillna(0)


In [ ]:
# --- Category diversity (view+cart only) ---
unique_categories = df_feat.groupby('user_session')['category_code'].nunique()
session_time['unique_categories'] = unique_categories.reindex(session_time.index).fillna(0)


In [ ]:
# --- Price features (view+cart only) ---
# Using df_feat ensures we only see prices of browsed items,
# NOT the price at the moment of purchase (which would be leakage).
price_stats = df_feat.groupby('user_session')['price'].agg(
    avg_price='mean',
    max_price='max',
    price_std='std'
).reindex(session_time.index).fillna(0)
session_time = session_time.join(price_stats)


In [ ]:
# --- Day of week (view+cart only) ---
df_feat['dayofweek'] = df_feat['event_time'].dt.dayofweek
session_time['dayofweek'] = (
    df_feat.groupby('user_session')['dayofweek']
    .agg(lambda x: x.mode().iloc[0])
).reindex(session_time.index).fillna(0)


In [ ]:
print(session_time.isnull().sum())
session_time.head()

## 5. Modelling with Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
features = [
    'session_duration',
    'session_length',
    'unique_products',
    'cart_count',
    'view_count',
    'cart_view_ratio',
    'avg_time_gap',
    'repeat_product_views',
    'unique_categories',
    'avg_price',
    'max_price',
    'price_std',
    'dayofweek',
]

X = session_time[features]
y = session_time['target']

print(f'Feature matrix: {X.shape}, class balance: {y.mean() * 100:.2f}% positive')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

lr_param_dist = {
    'C':       loguniform(1e-3, 1e2),   # inverse regularisation strength
    'solver':  ['lbfgs', 'saga'],
    'penalty': ['l2'],
}

lr_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_distributions=lr_param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
lr_search.fit(X_train_scaled, y_train)

print('Best LR params:', lr_search.best_params_)
print(f'Best CV ROC-AUC: {lr_search.best_score_:.4f}')

lr_best = lr_search.best_estimator_
lr_pred = lr_best.predict(X_test_scaled)
lr_prob = lr_best.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_param_dist = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [6, 8, 10, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_train, y_train)

print('Best RF params:', rf_search.best_params_)
print(f'Best CV ROC-AUC: {rf_search.best_score_:.4f}')

rf_best = rf_search.best_estimator_
rf_pred = rf_best.predict(X_test)
rf_prob = rf_best.predict_proba(X_test)[:, 1]

In [ ]:
from xgboost import XGBClassifier

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

xgb_param_dist = {
    'n_estimators':   [100, 200, 300],
    'max_depth':      [3, 4, 5, 6, 7],
    'learning_rate':  [0.01, 0.05, 0.1, 0.2],
    'subsample':      [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':          [0, 0.1, 0.3, 0.5],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=neg / pos,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ),
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
xgb_search.fit(X_train, y_train)

print('Best XGB params:', xgb_search.best_params_)
print(f'Best CV ROC-AUC: {xgb_search.best_score_:.4f}')

xgb_best = xgb_search.best_estimator_
xgb_pred = xgb_best.predict(X_test)
xgb_prob = xgb_best.predict_proba(X_test)[:, 1]

## 6. Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

print('=== Best Hyperparameters ===')
print(f'LR:  {lr_search.best_params_}')
print(f'RF:  {rf_search.best_params_}')
print(f'XGB: {xgb_search.best_params_}')
print()

model_results = {
    'Logistic Regression (tuned)': (lr_pred, lr_prob),
    'Random Forest (tuned)':       (rf_pred, rf_prob),
    'XGBoost (tuned)':             (xgb_pred, xgb_prob),
}

rows = []
for name, (pred, prob) in model_results.items():
    rows.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall':    round(recall_score(y_test, pred), 4),
        'F1':        round(f1_score(y_test, pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob), 4),
    })

comparison_df = pd.DataFrame(rows).set_index('Model')
print('=== Test Set Performance ===')
print(comparison_df.to_string())

In [ ]:
# --- Confusion matrices ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

import numpy as np

for ax, (name, (pred, _)) in zip(axes, model_results.items()):
    cm = confusion_matrix(y_test, pred)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['No Purchase', 'Purchase'])
    ax.set_yticklabels(['No Purchase', 'Purchase'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# --- ROC Curves ---
from sklearn.metrics import roc_curve

plt.figure(figsize=(7, 5))
colors = ['steelblue', 'darkorange', 'green']

for (name, (_, prob)), color in zip(model_results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color)

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Precision-Recall Curves ---
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(7, 5))

for (name, (_, prob)), color in zip(model_results.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    plt.plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=color)

baseline = y_test.mean()
plt.axhline(y=baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Cross-validation on best XGBoost model ---
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_best, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'XGBoost (tuned) 5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold: {[round(s, 4) for s in cv_scores]}')

## 7. Feature Importance & SHAP

In [ ]:
# --- XGBoost (tuned) feature importance ---
xgb_fi = pd.DataFrame({
    'Feature':    features,
    'Importance': xgb_best.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(xgb_fi['Feature'], xgb_fi['Importance'])
plt.xlabel('Importance Score')
plt.title('XGBoost (tuned) Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# --- SHAP summary plot ---
import shap

explainer   = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, feature_names=features, plot_type='dot')